### Predict (by hand)
Credited for handing it in, never marked right or wrong.

* **Qwen2.5-1.5B-Instruct** is about 1.5 billion parameters:
  * At **fp16** (2 bytes each) the weights alone are about **3.0** GB.
  * At **int8** (1 byte each) about **1.5** GB.
* **Resident VRAM:**
  * At **512 context, fp16**: **~4.0** GB.
  * At **4096 context, fp16**: **~6.0** GB.
  * *(Context 4096 is larger by roughly **2.0 GB** due to KV cache growth).*
* During a single-request decode (one prompt, generating tokens one at a time), GPU utilisation will read about **90-98** percent.

---

### Cell 1: install pins and the sampler

In [2]:
import csv, subprocess, sys, threading, time

# 1. تثبيت الحزم
VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"


def pip_install(*specs):
  cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
  print("installing:", " ".join(specs))
  subprocess.run(cmd, check=True)


pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"bitsandbytes=={BITSANDBYTES_PIN}",
)

# 2. تعريف دوال أخذ العينات (Sampler)
GPU_SAMPLES = "/content/gpu_samples.csv"
_sampler = {"thread": None, "stop": None}


def _sample_loop(stop_event, path, interval_s):
  with open(path, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["t", "util_gpu", "mem_used_mib"])
    t0 = time.time()
    while not stop_event.is_set():
      out = subprocess.run(
          [
              "nvidia-smi",
              "--query-gpu=utilization.gpu,memory.used",
              "--format=csv,noheader,nounits",
          ],
          capture_output=True,
          text=True,
      ).stdout.strip()
      parts = [p.strip() for p in out.split(",")]
      if len(parts) == 2:
        w.writerow([round(time.time() - t0, 2), parts[0], parts[1]])
        fh.flush()
      stop_event.wait(interval_s)


def start_sampler(path=GPU_SAMPLES, interval_s=2):
  if _sampler["thread"] and _sampler["thread"].is_alive():
    return
  stop = threading.Event()
  th = threading.Thread(
      target=_sample_loop, args=(stop, path, interval_s), daemon=True
  )
  th.start()
  _sampler["thread"], _sampler["stop"] = th, stop


def stop_sampler():
  if _sampler["stop"]:
    _sampler["stop"].set()
  if _sampler["thread"]:
    _sampler["thread"].join(timeout=5)
  _sampler["thread"], _sampler["stop"] = None, None


def read_util_mean(path=GPU_SAMPLES):
  vals = []
  try:
    with open(path) as fh:
      for row in csv.DictReader(fh):
        try:
          vals.append(float(row["util_gpu"]))
        except (KeyError, ValueError):
          pass
  except FileNotFoundError:
    return 0.0
  return sum(vals) / len(vals) if vals else 0.0


# 3. التأكد من كرت الشاشة
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

installing: transformers==4.46.* accelerate==1.1.* bitsandbytes==0.49.2
Tesla T4, 15360 MiB


### Cell 2: the measurement helper

In [3]:
import csv, gc, subprocess, threading, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- 1. تعريف دوال الـ Sampler ---
GPU_SAMPLES = "/content/gpu_samples.csv"
_sampler = {"thread": None, "stop": None}


def _sample_loop(stop_event, path, interval_s):
  with open(path, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["t", "util_gpu", "mem_used_mib"])
    t0 = time.time()
    while not stop_event.is_set():
      out = subprocess.run(
          [
              "nvidia-smi",
              "--query-gpu=utilization.gpu,memory.used",
              "--format=csv,noheader,nounits",
          ],
          capture_output=True,
          text=True,
      ).stdout.strip()
      parts = [p.strip() for p in out.split(",")]
      if len(parts) == 2:
        w.writerow([round(time.time() - t0, 2), parts[0], parts[1]])
        fh.flush()
      stop_event.wait(interval_s)


def start_sampler(path=GPU_SAMPLES, interval_s=2):
  if _sampler["thread"] and _sampler["thread"].is_alive():
    return
  stop = threading.Event()
  th = threading.Thread(
      target=_sample_loop, args=(stop, path, interval_s), daemon=True
  )
  th.start()
  _sampler["thread"], _sampler["stop"] = th, stop


def stop_sampler():
  if _sampler["stop"]:
    _sampler["stop"].set()
  if _sampler["thread"]:
    _sampler["thread"].join(timeout=5)
  _sampler["thread"], _sampler["stop"] = None, None


def read_util_mean(path=GPU_SAMPLES):
  vals = []
  try:
    with open(path) as fh:
      for row in csv.DictReader(fh):
        try:
          vals.append(float(row["util_gpu"]))
        except (KeyError, ValueError):
          pass
  except FileNotFoundError:
    return 0.0
  return sum(vals) / len(vals) if vals else 0.0


# --- 2. تحميل الموديل ودوال القياس ---
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"


def load(dtype: str):
  if dtype == "fp16":
    return AutoModelForCausalLM.from_pretrained(
        MODEL, torch_dtype=torch.float16, device_map="cuda"
    )
  if dtype == "int8":
    qc = BitsAndBytesConfig(load_in_8bit=True)
    return AutoModelForCausalLM.from_pretrained(
        MODEL,
        quantization_config=qc,
        torch_dtype=torch.float16,
        device_map="cuda",
    )
  raise ValueError(dtype)


def make_prompt(context_tokens: int) -> str:
  base = "Summarise the following text in one sentence.\n"
  filler = "The data center runs many small inference requests all day. " * 400
  ids = tok(base + filler)["input_ids"][:context_tokens]
  return tok.decode(ids)


def resident_vram_gb() -> float:
  torch.cuda.synchronize()
  return torch.cuda.memory_reserved() / (1024**3)


def profile(
    model, dtype: str, context: int, new_tokens: int = 128, batch: int = 1
):
  prompt = make_prompt(context)
  prompts = [prompt] * batch
  enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")

  # warm-up (compile/allocate)
  _ = model.generate(
      **enc,
      max_new_tokens=8,
      do_sample=False,
      temperature=None,
      top_p=None,
      top_k=None,
  )
  vram = resident_vram_gb()

  start_sampler()
  t0 = time.time()
  out = model.generate(
      **enc,
      max_new_tokens=new_tokens,
      do_sample=False,
      temperature=None,
      top_p=None,
      top_k=None,
  )
  dt = time.time() - t0
  stop_sampler()

  gen_tokens = (out.shape[1] - enc["input_ids"].shape[1]) * batch
  return {
      "dtype": dtype,
      "context": context,
      "vram_gb": round(vram, 3),
      "util_mean": round(read_util_mean(), 1),
      "tokens_per_s": round(gen_tokens / dt, 1),
  }


def free_vram():
  gc.collect()
  torch.cuda.empty_cache()

### Cell 3: the matrix

In [4]:
!pip install -U "transformers==4.46.3" "accelerate>=0.26.0" "bitsandbytes>=0.43.0"

  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached accelerate-1.14.0-py3-none-any.whl (389 kB)
Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl (43.1 MB)
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.49.2
    Uninstalling bitsandbytes-0.49.2:
      Successfully uninstalled bitsandbytes-0.49.2
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.1.1
    Uninstalling accelerate-1.1.1:
      Successfully uninstalled accelerate-1.1.1


In [4]:
rows = []
for dtype in ["fp16", "int8"]:
  model = load(dtype)
  for context in [512, 2048, 4096]:
    row = profile(model, dtype, context)
    print(row)
    rows.append(row)
  del model
  free_vram()

{'dtype': 'fp16', 'context': 512, 'vram_gb': 6.166, 'util_mean': 60.3, 'tokens_per_s': 29.8}
{'dtype': 'fp16', 'context': 2048, 'vram_gb': 6.355, 'util_mean': 67.3, 'tokens_per_s': 26.7}
{'dtype': 'fp16', 'context': 4096, 'vram_gb': 6.561, 'util_mean': 77.7, 'tokens_per_s': 25.0}
{'dtype': 'int8', 'context': 512, 'vram_gb': 4.854, 'util_mean': 26.9, 'tokens_per_s': 6.2}
{'dtype': 'int8', 'context': 2048, 'vram_gb': 5.082, 'util_mean': 27.5, 'tokens_per_s': 5.9}
{'dtype': 'int8', 'context': 4096, 'vram_gb': 5.355, 'util_mean': 32.8, 'tokens_per_s': 5.4}


### Cell 4: the utilisation-vs-busy experiment

In [5]:
model = load("fp16")
b1 = profile(model, "fp16", 512, new_tokens=128, batch=1)
b8 = profile(model, "fp16", 512, new_tokens=128, batch=8)
del model
free_vram()
print("batch 1:", b1)
print("batch 8:", b8)
print("tokens/s ratio:", round(b8["tokens_per_s"] / b1["tokens_per_s"], 2))
print("util delta:", round(b8["util_mean"] - b1["util_mean"], 1))

batch 1: {'dtype': 'fp16', 'context': 512, 'vram_gb': 6.166, 'util_mean': 62.3, 'tokens_per_s': 28.8}
batch 8: {'dtype': 'fp16', 'context': 512, 'vram_gb': 6.51, 'util_mean': 69.3, 'tokens_per_s': 175.3}
tokens/s ratio: 6.09
util delta: 7.0


In [6]:
import json
with open("batch_check.json", "w") as f:
    json.dump({"batch1_tokens_per_s": b1["tokens_per_s"],
               "batch8_tokens_per_s": b8["tokens_per_s"]}, f, indent=2)

### Cell 5: write profile.json

In [7]:
import json
with open("profile.json", "w") as f:
    json.dump(rows, f, indent=2)
print("wrote", len(rows), "rows to profile.json")

wrote 6 rows to profile.json


### Verify (green check) & Download Artifacts

In [8]:
# Green-check verifier for Lab W3D1 (profile inference).
# Paste this as the last cell of your day-1 notebook and run it. It reads
# profile.json (the matrix rows you wrote) and checks the schema and the sanity
# rules. It also reads the batch experiment numbers if you saved them to
# batch_check.json; if that file is absent it asks for the two numbers inline so
# the batch-8 > batch-1 rule can still be checked.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os

REQUIRED_KEYS = {"dtype", "context", "vram_gb", "util_mean", "tokens_per_s"}


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def load_json(path: str):
    if not os.path.exists(path):
        fail(f"{path} not found; write it in the last data cell")
    try:
        with open(path) as fh:
            return json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")


def main() -> None:
    rows = load_json("profile.json")

    if not isinstance(rows, list) or not rows:
        fail("profile.json must be a non-empty list of rows")

    # schema
    for i, row in enumerate(rows):
        if not isinstance(row, dict):
            fail(f"row {i} is not an object")
        missing = REQUIRED_KEYS - set(row)
        if missing:
            fail(f"row {i} missing keys: {sorted(missing)}")

    dtypes = {r["dtype"] for r in rows}
    contexts = sorted({r["context"] for r in rows})
    if "fp16" not in dtypes:
        fail("no fp16 rows; the matrix needs fp16")
    if len(contexts) < 3:
        fail(f"need at least 3 context lengths, found {contexts}")

    # sanity 1: VRAM rises with context (within each dtype)
    for dt in dtypes:
        sub = sorted((r for r in rows if r["dtype"] == dt),
                     key=lambda r: r["context"])
        vrams = [r["vram_gb"] for r in sub]
        if any(b < a - 0.01 for a, b in zip(vrams, vrams[1:])):
            fail(f"{dt} VRAM does not rise with context: {vrams}")

    # sanity 2: fp16 uses more memory than int8 at a shared context
    if "int8" in dtypes:
        shared = None
        for c in contexts:
            has_fp16 = any(r["dtype"] == "fp16" and r["context"] == c for r in rows)
            has_int8 = any(r["dtype"] == "int8" and r["context"] == c for r in rows)
            if has_fp16 and has_int8:
                shared = c
                break
        if shared is None:
            fail("fp16 and int8 share no context length to compare")
        fp16_v = next(r["vram_gb"] for r in rows
                      if r["dtype"] == "fp16" and r["context"] == shared)
        int8_v = next(r["vram_gb"] for r in rows
                      if r["dtype"] == "int8" and r["context"] == shared)
        if not fp16_v > int8_v:
            fail(f"fp16 VRAM ({fp16_v}) not above int8 VRAM ({int8_v}) at "
                 f"context {shared}")

    # sanity 3: batch-8 tokens/s beats batch-1
    b1 = b8 = None
    if os.path.exists("batch_check.json"):
        bc = load_json("batch_check.json")
        b1 = bc.get("batch1_tokens_per_s")
        b8 = bc.get("batch8_tokens_per_s")
    else:
        # allow the two numbers as module-level names set in an earlier cell
        b1 = globals().get("BATCH1_TOKENS_PER_S")
        b8 = globals().get("BATCH8_TOKENS_PER_S")
    if b1 is None or b8 is None:
        fail("batch numbers missing; save batch_check.json with "
             "batch1_tokens_per_s and batch8_tokens_per_s, or set "
             "BATCH1_TOKENS_PER_S / BATCH8_TOKENS_PER_S")
    if not b8 > b1:
        fail(f"batch-8 tokens/s ({b8}) not above batch-1 ({b1})")

    print(f"rows: {len(rows)}, dtypes: {sorted(dtypes)}, contexts: {contexts}")
    print(f"batch-1 tokens/s: {b1}, batch-8 tokens/s: {b8}")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


rows: 6, dtypes: ['fp16', 'int8'], contexts: [512, 2048, 4096]
batch-1 tokens/s: 28.8, batch-8 tokens/s: 175.3
GREEN CHECK: PASS


In [9]:
from google.colab import files
for f_ in ["profile.json", "batch_check.json"]:
    files.download(f_)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>